In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import gc
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. 전처리 완료한 CatBoost 전용 데이터 로드(인코딩X)
train_cat = pd.read_csv('/content/drive/MyDrive/ESAA/OB/project1/train_catboost1.csv')
test_cat = pd.read_csv('/content/drive/MyDrive/ESAA/OB/project1/test_catboost1.csv')

X_cat = train_cat.drop(columns=['Click'])
y = train_cat['Click']
categorical_columns = ["F01", "F02", "F05", "F07", "F08","F09", "F10", "F12", "F13", "F16", "F17", "F21", "F22", "F23", "F25", "F28", "F30", "F31", "F34", "F35", "F37", "F39"]

X_tr_cat, X_va_cat, y_tr, y_va = train_test_split(X_cat, y, test_size=0.2, stratify=y, random_state=42)

# 모델 학습 및 검증/테스트 예측값 뽑기
model_cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, cat_features=categorical_columns, eval_metric='AUC', random_seed=42, verbose=False)
model_cat.fit(X_tr_cat, y_tr, eval_set=(X_va_cat, y_va), early_stopping_rounds=50)

# 메타 모델에게 넘겨줄 20% 예측값과, 최종 Test 예측값 저장
cat_val_preds = model_cat.predict_proba(X_va_cat)[:, 1]
cat_test_preds = model_cat.predict_proba(test_cat)[:, 1]

print(f"CatBoost 검증 AUC: {roc_auc_score(y_va, cat_val_preds):.5f}")

# 램 확보를 위해 CatBoost 데이터 삭제
del train_cat, test_cat, X_cat, X_tr_cat, X_va_cat, model_cat
gc.collect()


# 1. LGBM 전용 수치형 데이터 로드
train_lgbm = pd.read_csv('/content/drive/MyDrive/ESAA/OB/project1/train_final.csv')
test_lgbm = pd.read_csv('/content/drive/MyDrive/ESAA/OB/project1/test_final.csv')

X_lgbm = train_lgbm.drop(columns=['Click'])

X_tr_lgbm, X_va_lgbm, _, _ = train_test_split(X_lgbm, y, test_size=0.2, stratify=y, random_state=42)

# 2. 찾아낸 최적의 파라미터 딕셔너리 정의
lgbm_best_params = {
    'learning_rate': 0.04423786044029868,
    'num_leaves': 155,
    'max_depth': 13,
    'min_child_samples': 76,
    'feature_fraction': 0.6700322150966446,
    'bagging_fraction': 0.883453460183011,
    'bagging_freq': 4,
    'lambda_l1': 3.342536329323218e-05,
    'lambda_l2': 3.1845692568661615e-07,
    'n_estimators': 2000, # 나무를 충분히 많이 심도록 추가 (early_stopping이 제어함)
    'random_state': 42,
    'n_jobs': -1 # CPU 코어를 전부 사용해서 속도 업!
}

# 3. 모델 생성
model_lgbm = LGBMClassifier(**lgbm_best_params)

# 4. 학습 시작 (조기 종료 설정 추가)
from lightgbm import early_stopping, log_evaluation

model_lgbm.fit(
    X_tr_lgbm, y_tr,
    eval_set=[(X_va_lgbm, y_va)],
    callbacks=[early_stopping(stopping_rounds=100), log_evaluation(100)]
)

# 5. 예측값 뽑기
lgbm_val_preds = model_lgbm.predict_proba(X_va_lgbm)[:, 1]
lgbm_test_preds = model_lgbm.predict_proba(test_lgbm)[:, 1]

print(f"최적화된 LGBM 검증 AUC: {roc_auc_score(y_va, lgbm_val_preds):.5f}")

# 램 확보
del train_lgbm, test_lgbm, X_lgbm, X_tr_lgbm, X_va_lgbm, model_lgbm
import gc
gc.collect()


print("\n스태킹")

# 3. Base 모델들의 예측값(20% 검증셋)을 모아 새로운 문제지
stacked_train = pd.DataFrame({
    'CatBoost': cat_val_preds,
    'LGBM': lgbm_val_preds
})

# Base 모델들의 Test 예측값을 모아서 최종 시험지.
stacked_test = pd.DataFrame({
    'CatBoost': cat_test_preds,
    'LGBM': lgbm_test_preds
})

# 선형 회귀 모델이 두 모델의 주장을 듣고 어떤 비율로 섞을지 학습
meta_model = LogisticRegression()
meta_model.fit(stacked_train, y_va)

# 스태킹 최종 성능 확인
meta_val_preds = meta_model.predict_proba(stacked_train)[:, 1]
meta_auc = roc_auc_score(y_va, meta_val_preds)
print(f"최종 스태킹 검증 AUC Score: {meta_auc:.5f}")
print(f"반영 비율 - CatBoost: {meta_model.coef_[0][0]:.4f} / LGBM: {meta_model.coef_[0][1]:.4f}")


# 4. 최종 Test 예측 및 저장
final_preds = meta_model.predict_proba(stacked_test)[:, 1]

import zipfile
zip_path = "/content/drive/MyDrive/ESAA/OB/project1/open.zip"
with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("sample_submission.csv") as f:
        submission = pd.read_csv(f)

submission['Click'] = final_preds
submission_path = "/content/drive/MyDrive/ESAA/OB/project1/stacking_fast_baseline.csv"
submission.to_csv(submission_path, index=False)

print(f"스태킹 앙상블 완료! 파일 경로:\n{submission_path}")

CatBoost 검증 AUC: 0.76398
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.6700322150966446, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6700322150966446
[LightGBM] [Warning] lambda_l2 is set=3.1845692568661615e-07, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.1845692568661615e-07
[LightGBM] [Warning] lambda_l1 is set=3.342536329323218e-05, reg_alpha=0.0 will be ignored. Current value: lambda_l1=3.342536329323218e-05
[LightGBM] [Warning] bagging_fraction is set=0.883453460183011, subsample=1.0 will be ignored. Current value: bagging_fraction=0.883453460183011
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.6700322150966446, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6700322150966446
[LightGBM] [Warning] lamb